## ERP Customers Silver Ttransformation

### 00. Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col
from datetime import datetime

### 01. Build today's folder name and read from bronze table

In [0]:
today = datetime.today().strftime("%Y_%m_%d")   # e.g. 2026_06_23
bronze_table = f"`abc_business-core`.bronze_{today}.erp_customers"
df = spark.table(bronze_table)

### 02. Trim all string fields

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

### 03. Customer ID Cleanup

In [0]:
df = df.withColumn(
    "cid",
    F.when(col("cid").startswith("NAS"),
           F.substring(col("cid"), 4, F.length(col("cid"))))
     .otherwise(col("cid"))
)

### 04. Birthdate Validation


In [0]:
df = df.withColumn(
    "bdate",
    F.when(col("bdate") > F.current_date(), None)
     .otherwise(col("bdate"))
)

### 05. Gender Normalization

In [0]:
df = df.withColumn(
    "gen",
    F.when(F.upper(col("gen")).isin("F", "FEMALE"), "Female")
     .when(F.upper(col("gen")).isin("M", "MALE"), "Male")
     .otherwise("n/a")
)

### 06. Renaming Columns

In [0]:
RENAME_MAP = {
    "cid": "customer_number",
    "bdate": "birth_date",
    "gen": "gender"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

### 07. Sanity checks of dataframe

In [0]:
df.limit(5).display()

### 08. Writing Silver Table

In [0]:
table_name = f"`abc_business-core`.silver.erp_customers_{today}"
df.write.mode("overwrite").format("delta").saveAsTable(table_name)

### 09. Sanity checks of silver table

In [0]:
df = spark.sql(f"SELECT * FROM {table_name} LIMIT 5")
df.show()